In [67]:
# Switch path to root of project
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
current_folder = globals()['_dh'][0]
os.chdir(os.path.dirname(os.path.abspath(current_folder)))
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [68]:
import cv2
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from eval.utils import RGBFIDDataset
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

from torchmetrics.image.fid import FrechetInceptionDistance
from torch.utils.data import Dataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"

In [69]:
first_rgb = Image.open("/scratch/tmaillar/rgb/train/point_32_view_0_domain_rgb.png").convert("RGB")
second_rgb = Image.open("/scratch/tmaillar/rgb/train/point_40_view_0_domain_rgb.png").convert("RGB")

crop_settings_0 = np.load("/work/com-304/datasets/clevr_com_304/train/crop_settings/00001.npy")

def format_image_for_FID(path, crop_settings, aug_idx = 1, device="cuda"):
    img = Image.open(path).convert("RGB")
    
    x1, y1, x2, y2, flip = crop_settings[aug_idx]
    top, left, h, w = y1, x1, (y2 - y1), (x2 - x1)
    
    cropped = TF.crop(img, top, left, h, w)
    resized = cropped.resize((256, 256), resample=Image.BILINEAR)
    
    if flip:
        resized = TF.hflip(resized)
        
    img = np.array(resized)
    img = torch.from_numpy(img).permute(2,0,1).byte()
    
    resized.show()

    return img.to(device).unsqueeze(0)



In [70]:
fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)

In [71]:
dataset = RGBFIDDataset(
    root_dir="/scratch/tmaillar/rgb/train",
    crop_settings=np.load("/work/com-304/datasets/clevr_com_304/train/crop_settings/00001.npy")
)

loader = DataLoader(dataset, batch_size=32, shuffle=False)

In [72]:
for imgs in loader:
    imgs = imgs.to(device)
    fid.update(imgs, real=True)

KeyboardInterrupt: 